# v28 後処理ラボ — validation(OOF)だけでチューニング → 提出で使う

**狙い**: v28 tabular(現状最強, 今は apply_pp のみ)に、**OOFで調整した後処理**を足す。testへの適合は保証しないが validation で最適化した値を提出でも使う。
- 学習済 `artifacts_v28`(models 9変種×5fold / features.json / pp_params.json / data/train.parquet)を使用。**再学習しない**。
- OOFアンサンブル予測を再構成 → apply_pp を baseline → **大量の後処理**を掃引。
- **複数指標**で見る: (1) overall RMSE, (2) **RMSE>10 の坑井数(pub/priv頑健の候補)**, (3) 坑井別中央値, (4) **改善の平坦さ/幅(ロバスト)**。
- 後処理族: 平滑(TVT空間) / **地形的=geo_z=TVT+Z 空間の平滑・robust射影・dip制限** / アンカーramp / PFブレンド / rate制限 …。
- 得た最適パラメータ → `pp_extra_params.json` 保存 + submit 用 apply 関数を出力。


In [ ]:
# ===== 読込 + OOF再構成 + Z/MD補完 + apply_pp baseline =====
import os, json, glob, warnings, joblib
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import GroupKFold
warnings.filterwarnings("ignore"); np.seterr(all="ignore")

ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v28\artifacts_v28")
PARQUET = ART / "data" / "train.parquet"
features = json.loads((ART / "features.json").read_text())
PP = json.loads((ART / "pp_params.json").read_text())
N_SPLITS = 5
print("PP:", PP, "| features:", len(features))

df = pd.read_parquet(PARQUET)
META = {"well", "id", "target", "pf_ancc"}
y  = df["target"].to_numpy(np.float64)
g  = df["well"].to_numpy()
lk = df["last_known_tvt"].to_numpy(np.float64)
mds = df["md_since"].to_numpy(np.float64) if "md_since" in df.columns else np.zeros(len(df))
pf_abs = df["pf_ancc"].to_numpy(np.float64) if "pf_ancc" in df.columns else lk.copy()
X = df[[c for c in features if c in df.columns]].astype("float32")
print("rows", len(df), "| wells", pd.unique(g).size)

# ---- OOF アンサンブル再構成(GroupKFold(5) by well, 学習と同一分割) ----
folds = list(GroupKFold(n_splits=N_SPLITS).split(X, y, groups=g))
variants = sorted([d.name for d in (ART / "models").glob("*") if d.is_dir()])
oof_list = []
for v in variants:
    o = np.full(len(y), np.nan); ok = True
    for k, (tr, va) in enumerate(folds):
        fp = ART / "models" / v / f"fold{k}.pkl"
        if not fp.exists(): ok = False; break
        o[va] = joblib.load(fp).predict(X.iloc[va])
    if ok and np.isfinite(o).all():
        oof_list.append(o); print(f"  OOF {v} ok")
    else:
        print(f"  OOF {v} SKIP(欠損)")
assert oof_list, "OOF再構成失敗(fold pkl 不足?)"
oof_ens = np.mean(oof_list, axis=0)
print("OOF variants used:", len(oof_list))

# ---- Z / MD を raw train CSV から eval行順で補完(地形的後処理用) ----
def _detect_train():
    for p in [r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction\train",
              "/kaggle/input/rogii-wellbore-geology-prediction/train"]:
        if Path(p).exists(): return Path(p)
    hits = glob.glob("/kaggle/input/**/train/*__horizontal_well.csv", recursive=True)
    return Path(hits[0]).parent if hits else None
TRAIN = _detect_train(); print("TRAIN dir:", TRAIN)

groups = {w: np.asarray(idx) for w, idx in df.groupby("well", sort=False).indices.items()}
Z = np.full(len(df), np.nan); MD = np.full(len(df), np.nan); nmiss = 0
for w, idx in groups.items():
    fp = TRAIN / f"{w}__horizontal_well.csv"
    if not fp.exists(): nmiss += 1; continue
    hw = pd.read_csv(fp); ev = hw[hw["TVT_input"].isna()]
    n = min(len(idx), len(ev))
    Z[idx[:n]] = ev["Z"].to_numpy(float)[:n]; MD[idx[:n]] = ev["MD"].to_numpy(float)[:n]
Z = np.where(np.isfinite(Z), Z, 0.0); MD = np.where(np.isfinite(MD), MD, mds)
print("Z/MD 補完: 欠損well", nmiss)

# ---- apply_pp baseline ----
w_pf = float(PP.get("w_pf", 0.09)); tau = PP.get("tau", 85); alpha = float(PP.get("alpha", 1.0))
pf_res = pf_abs - lk
def apply_pp(res):
    d = res * (1 - w_pf) + pf_res * w_pf
    if tau: d = d * (1.0 - np.exp(-np.maximum(mds, 0.0) / float(tau)))
    return d * alpha
pred0 = lk + apply_pp(oof_ens)          # baseline 予測 TVT
true_tvt = lk + y                        # 真値 TVT

# 追加で使える特徴(あれば)
def col(name): return df[name].to_numpy(np.float64) if name in df.columns else None
AGREE = col("pf_agree_std") or col("sm_pf_agree_std")
NNDIST = col("pf_nbr_nndist"); PSELF = col("pf_self"); PNBR = col("pf_nbr")
CTX_COLS = {k: v for k, v in dict(agree=AGREE, nndist=NNDIST, pself=PSELF, pnbr=PNBR).items() if v is not None}
print("追加ctx列:", list(CTX_COLS.keys()))

In [ ]:
# ===== 指標 + 坑井別スライス + 掃引エンジン =====
WIDS = list(groups.keys()); IDX = [groups[w] for w in WIDS]
LKW = [float(lk[ix][0]) for ix in IDX]                       # 坑井の last_known(定数)
def _perwell(arr): return [arr[ix] for ix in IDX]
MDS_W = _perwell(mds); PF_W = _perwell(pf_abs); Z_W = _perwell(Z); MD_W = _perwell(MD); TRUE_W = _perwell(true_tvt)
CTX_W = {k: _perwell(v) for k, v in CTX_COLS.items()}

def metrics(pred_full):
    e = pred_full - true_tvt; ok = np.isfinite(e)
    overall = float(np.sqrt(np.mean(e[ok] ** 2))) if ok.any() else 1e9
    pw = np.array([np.sqrt(np.mean((pred_full[ix] - true_tvt[ix]) ** 2)) for ix in IDX])
    return dict(overall=overall, n_over10=int((pw > 10).sum()), med_pw=float(np.median(pw)), mean_pw=float(np.mean(pw)))

def make_ctx(wi):
    ctx = dict(lk=LKW[wi], mds=MDS_W[wi], pf=PF_W[wi], z=Z_W[wi], md=MD_W[wi])
    for k in CTX_W: ctx[k] = CTX_W[k][wi]
    return ctx

def apply_perwell(fn, param):
    """fn(p, ctx, param) を全坑井に適用(blanket)。"""
    out = pred0.copy()
    for wi, ix in enumerate(IDX):
        try: out[ix] = fn(pred0[ix].copy(), make_ctx(wi), param)
        except Exception: pass
    return out

BASE = metrics(pred0)
print("baseline(apply_ppのみ):", {k: round(v, 4) if isinstance(v, float) else v for k, v in BASE.items()})

def robust_pick(params, overalls, tol=0.03):
    """min近傍(min+tol)で最も広い連続帯の中心paramを"ロバスト解"に。平坦で低い=転移候補。"""
    o = np.asarray(overalls, float); mn = np.nanmin(o); flat = o <= mn + tol
    best_i, best_len, i = 0, 0, 0
    while i < len(flat):
        if flat[i]:
            j = i
            while j + 1 < len(flat) and flat[j + 1]: j += 1
            if j - i + 1 > best_len: best_len, best_i = j - i + 1, (i + j) // 2
            i = j + 1
        else: i += 1
    return params[best_i], int(best_len), float(mn)

def sweep(name, fn, grid):
    rows = []
    for p in grid:
        m = metrics(apply_perwell(fn, p)); m["param"] = p; rows.append(m)
    r = pd.DataFrame(rows)
    rp, wid, mn = robust_pick(list(r["param"]), list(r["overall"]))
    best_rmse = r.loc[r["overall"].idxmin()]; best_over = r.loc[r["n_over10"].idxmin()]
    SUMMARY.append(dict(name=name,
        best_rmse_param=best_rmse["param"], best_rmse=round(float(best_rmse["overall"]), 4),
        d_rmse=round(float(best_rmse["overall"] - BASE["overall"]), 4),
        best_over10_param=best_over["param"], min_over10=int(best_over["n_over10"]),
        d_over10=int(best_over["n_over10"] - BASE["n_over10"]),
        robust_param=rp, robust_flatN=wid, robust_rmse=round(float(mn), 4)))
    RESULTS[name] = r
    return r
SUMMARY = []; RESULTS = {}

In [ ]:
# ===== 後処理ライブラリ(大量) =====
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter1d, median_filter

def _odd(w, n): w = int(min(w, n)); return w - 1 if w % 2 == 0 else w
def _sg(p, w, q):
    n = len(p); wl = _odd(w, n)
    return savgol_filter(p, wl, q) if wl >= q + 2 else p
def _robfit(s, ycol, deg=4, iters=4):
    if len(s) < deg + 2: return ycol.copy()
    c = np.polyfit(s, ycol, deg)
    for _ in range(iters):
        r = ycol - np.polyval(c, s); sc = np.median(np.abs(r)) * 1.4826 + 1e-6
        c = np.polyfit(s, ycol, deg, w=1.0 / (1.0 + (r / (2 * sc)) ** 2))
    return np.polyval(c, s)
def _snorm(mds):
    a = np.asarray(mds, float); rng = max(a.max() - a.min(), 1e-6); return (a - a.min()) / rng
def _ema_fb(p, a):
    f = p.copy()
    for i in range(1, len(f)): f[i] = a * p[i] + (1 - a) * f[i - 1]
    b = p.copy()
    for i in range(len(b) - 2, -1, -1): b[i] = a * p[i] + (1 - a) * b[i + 1]
    return 0.5 * (f + b)
def _ratelimit(p, mds, rmax):
    q = p.copy()
    for i in range(1, len(q)):
        dm = max(mds[i] - mds[i - 1], 1e-6); mx = rmax * dm
        q[i] = np.clip(q[i], q[i - 1] - mx, q[i - 1] + mx)
    return q

# --- 各後処理: fn(p, ctx, param) -> new p ---
PP_LIB = {}
def reg(name, grid):
    def deco(f): PP_LIB[name] = (f, grid); return f
    return deco

# 平滑(TVT空間)
reg("sg_w",           [5,7,9,13,17,21,25,31])(lambda p,c,w: _sg(p,w,3))
reg("median_w",       [3,5,7,9,13,17,21])(lambda p,c,w: median_filter(p, size=_odd(w,len(p))))
reg("gauss_sigma",    [0.5,1,1.5,2,3,4,6,8])(lambda p,c,s: gaussian_filter1d(p, s))
reg("ema_alpha",      [0.1,0.2,0.3,0.4,0.5,0.7,0.9])(lambda p,c,a: _ema_fb(p,a))
# 地形的(geo_z=TVT+Z 空間) — 層序位置は滑らか
reg("geo_sg_w",       [5,7,9,13,17,21,25,31])(lambda p,c,w: _sg(p+c["z"], w, 3) - c["z"])
reg("geo_gauss_sigma",[0.5,1,1.5,2,3,4,6,8])(lambda p,c,s: gaussian_filter1d(p+c["z"], s) - c["z"])
reg("geo_robfit_blend",[0.0,0.15,0.25,0.4,0.5,0.65,0.75,0.9,1.0])(
    lambda p,c,b: (1-b)*p + b*(_robfit(_snorm(c["mds"]), p+c["z"], 4) - c["z"]))
reg("geo_robfit_deg", [1,2,3,4,5,6])(
    lambda p,c,d: 0.25*p + 0.75*(_robfit(_snorm(c["mds"]), p+c["z"], int(d)) - c["z"]))
reg("geo_ratelimit",  [0.05,0.1,0.2,0.3,0.5,0.8,1.5,3.0])(       # dip(d geo_z/d md) の変化制限
    lambda p,c,r: _ratelimit(p+c["z"], c["md"] if np.isfinite(c["md"]).all() else c["mds"], r) - c["z"])
# アンカー / ramp
reg("hold_tau",       [20,40,60,85,120,180,260,400])(
    lambda p,c,t: c["lk"] + (p-c["lk"])*(1-np.exp(-np.maximum(c["mds"],0)/t)))
reg("shrink_s",       [0.80,0.88,0.92,0.95,0.98,1.0,1.02,1.05])(lambda p,c,s: c["lk"] + (p-c["lk"])*s)
reg("hold_w",         [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0])(lambda p,c,h: (1-h)*p + h*c["lk"])  # ヒューリスティックのhold
reg("delta_cap",      [10,20,30,40,60,80,120,200])(lambda p,c,cap: c["lk"] + np.clip(p-c["lk"], -cap, cap))
reg("ratelimit_tvt",  [0.05,0.1,0.2,0.3,0.5,0.8,1.5,3.0])(
    lambda p,c,r: _ratelimit(p, c["md"] if np.isfinite(c["md"]).all() else c["mds"], r))
# PF ブレンド / クランプ
reg("blend_pf_w",     [0.0,0.05,0.1,0.15,0.2,0.3,0.45,0.6])(lambda p,c,w: (1-w)*p + w*c["pf"])
reg("clamp_pf_band",  [200,120,80,50,30,20,12,8])(lambda p,c,b: np.clip(p, c["pf"]-b, c["pf"]+b))
PP_LIB_KEYS = list(PP_LIB.keys())
# 追加ctx(あれば): 不確実性で PF に寄せる
if "agree" in CTX_W:
    def _conf(p,c,a):
        ag = np.asarray(c["agree"],float); wgt = a*(ag/(np.nanmedian(ag)+1e-6)); wgt=np.clip(wgt,0,1)
        return (1-wgt)*p + wgt*c["pf"]
    reg("conf_blend_pf", [0.1,0.2,0.35,0.5,0.7,1.0])(_conf)
print("後処理数:", len(PP_LIB), "->", list(PP_LIB.keys()))

In [ ]:
# ===== well単位の信号 + ゲート付き後処理(条件を満たすwellだけに適用) =====
# 「全wellに波及」でなく「リスクの高い上位X%wellだけ rescue」。閾値(=条件)を調整。testでも同じ絶対閾値で適用可。
def _s_devpf(ix):  return float(np.nanmean(np.abs(pred0[ix] - pf_abs[ix])))     # GBM予測 vs PF の不一致
def _s_dmag(ix):   return float(abs(np.nanmean(pred0[ix] - lk[ix])))            # last_knownからの移動量
def _s_rough(ix):  return float(np.nanstd(np.diff(pred0[ix], 2))) if len(ix) > 3 else 0.0  # ジッタ/曲率
def _s_mdmax(ix):  return float(np.nanmax(mds[ix])) if len(ix) else 0.0         # 外挿の長さ
def _s_pfmax(ix):  return float(np.nanmax(np.abs(pred0[ix] - pf_abs[ix]))) if len(ix) else 0.0
SIGNALS = {"dev_pf": np.array([_s_devpf(ix) for ix in IDX]),
           "dmag":   np.array([_s_dmag(ix)  for ix in IDX]),
           "rough":  np.array([_s_rough(ix) for ix in IDX]),
           "mdmax":  np.array([_s_mdmax(ix) for ix in IDX]),
           "pfmax":  np.array([_s_pfmax(ix) for ix in IDX])}
if "agree" in CTX_W:  SIGNALS["agree"]  = np.array([float(np.nanmean(CTX_W["agree"][wi]))  for wi in range(len(IDX))])
if "nndist" in CTX_W: SIGNALS["nndist"] = np.array([float(np.nanmedian(CTX_W["nndist"][wi])) for wi in range(len(IDX))])
print("well信号:", list(SIGNALS.keys()))

def gated_apply(tname, param, sig_name, thr):
    """signal>=thr の well だけに後処理 tname を適用、他は baseline のまま。"""
    fn = PP_LIB[tname][0]; sv = SIGNALS[sig_name]; out = pred0.copy()
    for wi, ix in enumerate(IDX):
        if sv[wi] >= thr:
            try: out[ix] = fn(pred0[ix].copy(), make_ctx(wi), param)
            except Exception: pass
    return out

SUMMARY_GATED = []; RESULTS_GATED = {}
def gated_sweep(sig_name, tname, param, fracs=(0.03,0.05,0.1,0.15,0.2,0.3,0.5)):
    sv = SIGNALS[sig_name]; thrs = np.quantile(sv, [1 - f for f in fracs])
    rows = []
    for f, thr in zip(fracs, thrs):
        m = metrics(gated_apply(tname, param, sig_name, float(thr)))
        m.update(frac=f, thr=float(thr), n_gated=int((sv >= thr).sum())); rows.append(m)
    r = pd.DataFrame(rows); key = f"{sig_name}|{tname}|{param}"
    RESULTS_GATED[key] = r
    b_r = r.loc[r["overall"].idxmin()]; b_o = r.loc[r["n_over10"].idxmin()]
    SUMMARY_GATED.append(dict(signal=sig_name, pp=tname, param=param,
        best_rmse_frac=float(b_r["frac"]), best_rmse=round(float(b_r["overall"]), 4),
        d_rmse=round(float(b_r["overall"] - BASE["overall"]), 4),
        best_over10_frac=float(b_o["frac"]), min_over10=int(b_o["n_over10"]),
        d_over10=int(b_o["n_over10"] - BASE["n_over10"]), thr_at_over10=round(float(b_o["thr"]), 4)))
    return r

In [ ]:
# ===== 診断: 誤差解剖(どの後処理に頭打ちがあるか証拠ベースで) =====
def _lowfit(s, y, deg=3):
    if len(s) < deg + 2: return np.full_like(np.asarray(y, float), float(np.mean(y)))
    return np.polyval(np.polyfit(s, y, deg), s)
def _snz(m):
    m = np.asarray(m, float); r = max(m.max() - m.min(), 1e-6); return (m - m.min()) / r
def _rough(a): return float(np.std(np.diff(a, 2))) if len(a) > 3 else 0.0
def _overall(pred):
    e = (pred - true_tvt); e = e[np.isfinite(e)]; return float(np.sqrt(np.mean(e ** 2)))
Nrows = len(pred0)

# --- 1) 坑井別 誤差分解: bias / dip(低次) / jitter(高周波), flip候補 ---
sse_b = sse_d = sse_j = 0.0; flip = 0; flip_sse = 0.0
tot_sse = float(np.nansum((pred0 - true_tvt) ** 2))
for ix in IDX:
    e = pred0[ix] - true_tvt[ix]; n = e.size
    if n == 0: continue
    s = _snz(mds[ix]); b = float(np.nanmean(e)); sh = e - b
    lo = _lowfit(s, sh, 3); ji = sh - lo
    sse_b += b * b * n; sse_d += float(np.nansum(lo ** 2)); sse_j += float(np.nansum(ji ** 2))
    if abs(b) > 8.0 and (b * b * n) / (np.nansum(e ** 2) + 1e-9) > 0.6:
        flip += 1; flip_sse += float(np.nansum(e ** 2))
tot = sse_b + sse_d + sse_j + 1e-9
print("=== 1) 誤差分解(全プールSSE内訳) ===")
print(f"  bias(定数offset) {100*sse_b/tot:5.1f}%  RMSE寄与 {np.sqrt(sse_b/Nrows):.3f}  → 真値なしでは直せない(アンカー/PF較正要)")
print(f"  dip(傾き/低次)   {100*sse_d/tot:5.1f}%  RMSE寄与 {np.sqrt(sse_d/Nrows):.3f}  → geo低次fit / rate制限が効く")
print(f"  jitter(高周波)   {100*sse_j/tot:5.1f}%  RMSE寄与 {np.sqrt(sse_j/Nrows):.3f}  → 平滑が効く")
print(f"  flip候補well(双峰疑い): {flip}本  二乗誤差シェア {100*flip_sse/tot_sse:.1f}%  → hedge(PF/last_knownへ寄せる)のみ, 平滑は逆効果")

# --- 2) oracle 天井(各族のblind上限の目安。真値使用=到達不能) ---
def _oracle_blend(target_full):
    out = pred0.copy(); bs = np.linspace(0, 1, 11)
    for ix in IDX:
        tw = true_tvt[ix]; errs = [np.nanmean(((1 - b) * pred0[ix] + b * target_full[ix] - tw) ** 2) for b in bs]
        b = bs[int(np.argmin(errs))]; out[ix] = (1 - b) * pred0[ix] + b * target_full[ix]
    return out
p_bias = pred0.copy()
for ix in IDX: p_bias[ix] = pred0[ix] - np.nanmean(pred0[ix] - true_tvt[ix])
p_smT = pred0.copy(); p_gfit = pred0.copy()
for ix in IDX:
    s = _snz(mds[ix])
    if len(s) > 7:
        p_smT[ix] = np.polyval(np.polyfit(s, true_tvt[ix] + Z[ix], 6), s) - Z[ix]          # 真値geo_z deg6
        p_gfit[ix] = np.polyval(np.polyfit(s, pred0[ix] + Z[ix], 4), s) - Z[ix]            # pred geo_z deg4
print("\n=== 2) oracle 天井(到達不能の目安) ===")
print(f"  baseline(apply_pp)                 : {BASE['overall']:.3f}")
print(f"  oracle per-well bias除去           : {_overall(p_bias):.3f}  (offsetを完全に直せた床=biasの大きさ)")
print(f"  oracle 真値geo_z deg6(smooth oracle): {_overall(p_smT):.3f}  (§2 ≈3.0=絶対床)")
print(f"  oracle geo robfit(pred,best blend) : {_overall(_oracle_blend(p_gfit)):.3f}  (geo射影族の頭打ち)")
print(f"  oracle blend->last_known(best/well): {_overall(_oracle_blend(np.where(True, lk, lk))):.3f}  (hold hedge頭打ち)")
print(f"  oracle blend->pf_ancc(best/well)   : {_overall(_oracle_blend(pf_abs)):.3f}  (PF hedge頭打ち)")

# --- 3) ゲート信号の分離力(RMSE>10 を当てられるか = AUC) ---
PW = np.array([np.sqrt(np.nanmean((pred0[ix] - true_tvt[ix]) ** 2)) for ix in IDX]); bad = (PW > 10).astype(int)
def _auc(sig, lab):
    sig = np.asarray(sig, float); order = np.argsort(sig); rnk = np.empty(len(sig)); rnk[order] = np.arange(len(sig))
    n1 = int(lab.sum()); n0 = len(lab) - n1
    return float((rnk[lab == 1].sum() - n1 * (n1 - 1) / 2) / (n1 * n0)) if n1 and n0 else float("nan")
print(f"\n=== 3) ゲート信号 ↔ RMSE>10({int(bad.sum())}本) の分離力 ===")
if "SIGNALS" in globals():
    for name, sv in sorted(SIGNALS.items(), key=lambda kv: -_auc(kv[1], bad)):
        print(f"  {name:8s}: AUC(>10)={_auc(sv, bad):.3f}  corr(RMSE)={np.corrcoef(np.asarray(sv,float), PW)[0,1]:+.3f}")
else:
    print("  (SIGNALS 未定義=先にゲートセルを実行)")

# --- 4) 粗さ pred vs truth(geo平滑の生死) ---
rgp = np.median([_rough(pred0[ix] + Z[ix]) for ix in IDX]); rgt = np.median([_rough(true_tvt[ix] + Z[ix]) for ix in IDX])
rtp = np.median([_rough(pred0[ix]) for ix in IDX]); rtt = np.median([_rough(true_tvt[ix]) for ix in IDX])
print("\n=== 4) 粗さ(2次差分std中央値) pred vs truth ===")
print(f"  geo_z: pred={rgp:.3f} truth={rgt:.3f}  → {'predが粗い=geo平滑に余地' if rgp > rgt*1.15 else 'predは十分滑らか=geo平滑の余地小'}")
print(f"  TVT  : pred={rtp:.3f} truth={rtt:.3f}")

# --- 5) 誤差 vs md_since(外挿長 → ramp/hold の正当化) ---
ae = np.abs(pred0 - true_tvt); mfin = np.isfinite(mds) & np.isfinite(ae)
qs = np.quantile(mds[mfin], [0, .2, .4, .6, .8, 1.0])
print("\n=== 5) 誤差 vs md_since ===")
for i in range(5):
    m = (mds >= qs[i]) & (mds <= qs[i + 1]) & mfin
    print(f"  md_since [{qs[i]:6.0f},{qs[i+1]:6.0f}]: RMSE={np.sqrt(np.mean(ae[m]**2)):.3f}  (n={int(m.sum())})")
print("\n※ oracle は真値使用=到達不能。blind後処理の頭打ちの目安。bias支配なら平滑/rate系は無力、flip多なら hedge、jitter/dip があれば geo平滑/robfit。")

In [ ]:
# ===== 全掃引(blanket=全well) =====
import time
SUMMARY.clear(); RESULTS.clear()
t0 = time.time()
for name, (fn, grid) in PP_LIB.items():
    sweep(name, fn, grid)
    print(f"  swept {name} ({time.time()-t0:.0f}s)")
S = pd.DataFrame(SUMMARY)
print(f"\n=== baseline overall={BASE['overall']:.4f}  RMSE>10={BASE['n_over10']} ===")

In [ ]:
# ===== レポート: 複数観点でランキング =====
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_rows", 100)
print("■ overall RMSE 改善が大きい順(d_rmse昇順=最も下げた)")
print(S.sort_values("d_rmse")[["name","best_rmse_param","best_rmse","d_rmse","robust_param","robust_flatN","robust_rmse"]].to_string(index=False))
print("\n■ RMSE>10 本数を最も減らす順(pub/priv頑健の候補)")
print(S.sort_values(["min_over10","d_over10"])[["name","best_over10_param","min_over10","d_over10","best_rmse_param","best_rmse"]].to_string(index=False))
print("\n■ ロバスト(平坦な低RMSE帯が広い)順 — robust_flatN大 & robust_rmse低")
print(S.assign(score=S["robust_flatN"]*100 - S["robust_rmse"]).sort_values("score", ascending=False)[
    ["name","robust_param","robust_flatN","robust_rmse","best_rmse"]].to_string(index=False))

In [ ]:
# ===== 各後処理の掃引カーブ(param -> overall / n_over10) =====
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    ncol = 3; nrow = int(np.ceil(len(RESULTS)/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.2*nrow)); axes = np.atleast_1d(axes).ravel()
    for ax, (name, r) in zip(axes, RESULTS.items()):
        x = np.arange(len(r)); ax.plot(x, r["overall"], "-o", ms=3, label="overall")
        ax.axhline(BASE["overall"], color="gray", ls="--", lw=1)
        ax2 = ax.twinx(); ax2.plot(x, r["n_over10"], "-s", ms=3, color="tab:red", alpha=.5, label=">10")
        ax2.axhline(BASE["n_over10"], color="salmon", ls=":", lw=1)
        ax.set_title(name, fontsize=9); ax.set_xticks(x); ax.set_xticklabels([str(p) for p in r["param"]], rotation=45, fontsize=7)
    for ax in axes[len(RESULTS):]: ax.axis("off")
    plt.tight_layout(); fig.savefig(str(ART.parent/"pp_lab_curves.png"), dpi=110)
    print("saved", ART.parent/"pp_lab_curves.png")
except Exception as e:
    print("plot skip:", e)

In [ ]:
# ===== ゲート付き掃引(信号 × rescue後処理 × 上位frac) =====
import itertools, time
SUMMARY_GATED.clear(); RESULTS_GATED.clear()
# rescue系の後処理を、各wellレベル信号で「上位frac」だけに当てる(paramは強めの救済値)
GATE_PPS = [("hold_w",0.5),("hold_w",0.8),("shrink_s",0.9),("blend_pf_w",0.3),("blend_pf_w",0.5),
            ("clamp_pf_band",30),("clamp_pf_band",15),("geo_robfit_blend",0.5),("geo_sg_w",17)]
SIG_KEYS = list(SIGNALS.keys())
t0 = time.time()
for sig, (tname, param) in itertools.product(SIG_KEYS, GATE_PPS):
    gated_sweep(sig, tname, param)
print(f"gated sweeps done: {len(SUMMARY_GATED)} combos ({time.time()-t0:.0f}s)")
G = pd.DataFrame(SUMMARY_GATED)

In [ ]:
# ===== ゲート付きレポート(条件付き後処理の効き) =====
pd.set_option("display.width", 220, "display.max_rows", 120)
print(f"baseline overall={BASE['overall']:.4f}  RMSE>10={BASE['n_over10']}\n")
print("■ [ゲート] RMSE>10 を最も減らす順(条件付き=これが本命候補)")
print(G.sort_values(["min_over10","d_rmse"]).head(25)[
    ["signal","pp","param","best_over10_frac","min_over10","d_over10","thr_at_over10","best_rmse","d_rmse"]].to_string(index=False))
print("\n■ [ゲート] overall RMSE を最も下げる順")
print(G.sort_values("d_rmse").head(25)[
    ["signal","pp","param","best_rmse_frac","best_rmse","d_rmse","min_over10","d_over10"]].to_string(index=False))
print("\n※ frac=上位何%のwellに適用したか。thr=その絶対閾値(submitでは signal>=thr のwellに同処理)。")
G.to_csv(ART.parent / "pp_lab_gated_summary.csv", index=False); print("saved", ART.parent / "pp_lab_gated_summary.csv")

In [ ]:
# ===== 合成 + 保存(validationで選んだ後処理を順に適用 → 評価 → pp_extra_params.json) =====
# ★上のレポートを見て CHOSEN を編集。各stepは:
#   blanket:  {"pp": 名前, "param": 値}                      … 全wellに適用
#   gated:    {"pp": 名前, "param": 値, "gate": 信号, "thr": 閾値}  … signal>=thr のwellだけ
CHOSEN = [
    # 例(条件付きrescue): {"pp":"hold_w","param":0.5,"gate":"dev_pf","thr": 8.0},
    # 例(blanket軽平滑):  {"pp":"geo_sg_w","param":13},
]
def apply_chain(pred, chain):
    out = pred.copy()
    for st in chain:
        fn = PP_LIB[st["pp"]][0]; prm = st["param"]
        gate = st.get("gate"); thr = st.get("thr")
        sv = SIGNALS[gate] if gate else None
        for wi, ix in enumerate(IDX):
            if sv is not None and sv[wi] < thr: continue
            try: out[ix] = fn(out[ix].copy(), make_ctx(wi), prm)
            except Exception: pass
    return out
if CHOSEN:
    m = metrics(apply_chain(pred0, CHOSEN))
    print("baseline:", {k:round(v,4) if isinstance(v,float) else v for k,v in BASE.items()})
    print("chosen  :", {k:round(v,4) if isinstance(v,float) else v for k,v in m.items()})
    (ART / "pp_extra_params.json").write_text(json.dumps({"chain": CHOSEN}, ensure_ascii=False, indent=2))
    print("saved", ART / "pp_extra_params.json")
    print("# submit移植: apply_pp後のTVTに、上の apply_chain(+SIGNALS/信号定義)を per-well 適用。gateはtest実在列(pred/pf/features)のみ使用=test適用可。")
else:
    print("CHOSEN を上のレポート(特にゲート付き)を見て編集し、再実行してください。")
S.to_csv(ART.parent / "pp_lab_summary.csv", index=False); print("saved", ART.parent / "pp_lab_summary.csv")